### np.convolve

In [2]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _to_1d_list(x):
    shape = get_shape(x)
    if len(shape) == 0:
        return [x]
    if len(shape) != 1:
        raise ValueError("np.convolve only supports 1-D sequences")
    return x[:]


def np_convolve(a, v, mode='full'):
    a = _to_1d_list(a)
    v = _to_1d_list(v)

    if len(a) == 0 or len(v) == 0:
        return []

    # NumPy swaps if v is longer than a
    if len(v) > len(a):
        a, v = v, a

    n = len(a)
    m = len(v)

    full_len = n + m - 1
    full = [0] * full_len

    # convolution flips v
    vr = v[::-1]

    for i in range(full_len):
        total = 0
        for j in range(m):
            ai = i - j
            if 0 <= ai < n:
                total += a[ai] * vr[j]
        full[i] = total

    if mode == 'full':
        return full

    if mode == 'same':
        target_len = max(n, m)
        start = (full_len - target_len) // 2
        return full[start:start + target_len]

    if mode == 'valid':
        target_len = abs(n - m) + 1
        start = m - 1
        return full[start:start + target_len]

    raise ValueError("mode must be 'full', 'same', or 'valid'")

### test cases 

In [3]:
print(np_convolve([1, 2, 3], [0, 1, 0.5]))
print(np_convolve([1, 2, 3], [0, 1, 0.5], mode='same'))
print(np_convolve([1, 2, 3], [0, 1, 0.5], mode='valid'))
print(np_convolve([1, 2, 3, 4], [1, 1]))
print(np_convolve([1, 2, 3, 4], [1, 1], mode='same'))
print(np_convolve([1, 2, 3, 4], [1, 1], mode='valid'))
print(np_convolve([1], [2, 3, 4]))
print(np_convolve([1, 2, 3], [1])) 

[0.5, 2.0, 3.5, 3, 0]
[2.0, 3.5, 3]
[3.5]
[1, 3, 5, 7, 4]
[1, 3, 5, 7]
[3, 5, 7]
[2, 3, 4]
[1, 2, 3]


In [4]:
print(np_convolve([[1, 2], [3, 4]], [1, 2])) 

ValueError: np.convolve only supports 1-D sequences

In [5]:
print(np_convolve([1, 2, 3], [[1, 2]])) 

ValueError: np.convolve only supports 1-D sequences

In [6]:
print(np_convolve([1, 2, 3], [1, 2], mode='bad')) 

ValueError: mode must be 'full', 'same', or 'valid'

In [9]:
print(np_convolve([1, 2, 3], [])) 

[]
